# 01 — Extract & Load

**Purpose:** Extract raw monthly ZIP files, concatenate all CSVs into a single DataFrame, merge BTS lookup tables, and save the result as a Parquet file for downstream use.

**Input:**
- `data/*.zip` — raw monthly flight data (Jan–Dec 2024)
- `L_MONTHS.csv` — month code lookup
- `L_WEEKDAYS.csv` — weekday code lookup
- `L_UNIQUE_CARRIERS.csv` — airline carrier lookup

**Output:**
- `data/flights_2024.parquet` — combined raw flight data with decoded labels

**Key columns produced:**

| Column | Description |
|---|---|
| `YEAR`, `MONTH`, `DAY_OF_WEEK` | Date dimension codes from BTS source |
| `OP_UNIQUE_CARRIER` | Airline IATA code (e.g. `AA`) |
| `ORIGIN`, `DEST` | Airport IATA codes |
| `CRS_DEP_TIME` | Scheduled departure time in HHMM format |
| `DEP_DELAY` | Departure delay in minutes (negative = early) |
| `DEP_DEL15` | 1 if delayed 15+ minutes, else 0 |
| `CARRIER/WEATHER/NAS/SECURITY/LATE_AIRCRAFT_DELAY` | Delay minutes by cause |
| `MONTH_NAME`, `DAY`, `AIRLINE` | Human-readable labels decoded from lookup CSVs |
| `source_file` | Source ZIP name — used for traceability |

## Imports

In [ ]:
# Add imports here
from pathlib import Path
import zipfile
import logging
import pandas as pd

import sys
sys.path.append("..")   # or "." if running from data/ folder
from utils.logger import get_logger

## Extract ZIPs

In [ ]:
logging = get_logger()                                                                                                                                                                                                   

zip_folder = Path("unzip_data") # Where you save your raw data
extracted_folder = Path("extracted_files") # Where you save unzipped files

extracted_folder.mkdir(exist_ok=True)

zip_files = list(zip_folder.glob("*.zip"))
logging.info(f"Found {len(zip_files)} ZIP files")

# Unzip all monthly archives into extracted_files/
for zip_path in zip_files:
    target_folder = extracted_folder/zip_path.stem

    # If folder was created, skip to the next zip (avoid re-extract)
    if target_folder.exists():
        logging.info(f"Already extracted: {zip_path.name}, skipping")
        continue

    target_folder.mkdir()
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(target_folder)

    logging.info(f"Extracted: {zip_path.name} -> {target_folder}")


2026-04-09 16:01:17,258 - Found 12 ZIP files
2026-04-09 16:01:17,259 - Already extracted: july-2024.zip, skipping
2026-04-09 16:01:17,260 - Already extracted: sep-2024.zip, skipping
2026-04-09 16:01:17,260 - Already extracted: may-2024.zip, skipping
2026-04-09 16:01:17,261 - Already extracted: march-2024.zip, skipping
2026-04-09 16:01:17,261 - Already extracted: april-2024.zip, skipping
2026-04-09 16:01:17,262 - Already extracted: dec-2024.zip, skipping
2026-04-09 16:01:17,262 - Already extracted: aug-2024.zip, skipping
2026-04-09 16:01:17,262 - Already extracted: nov-2024.zip, skipping
2026-04-09 16:01:17,263 - Already extracted: jan-2024.zip, skipping
2026-04-09 16:01:17,263 - Already extracted: feb-2024.zip, skipping
2026-04-09 16:01:17,264 - Already extracted: oct-2024.zip, skipping
2026-04-09 16:01:17,264 - Already extracted: jun-2024.zip, skipping


## Load & Concatenate CSVs

In [21]:
# Read each extracted CSV and concatenate into one DataFrame
csv_files = list(extracted_folder.glob("*/*.csv")) # Find all CSVs in subfolders
logging.info(f"Found {len(csv_files)} CSV files")

dfs = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    df["source_file"] = csv_path.parent.name #track which month each row came from
    dfs.append(df)
    logging.info(f"Loaded: {csv_path.parent.name} - {len(df):,} rows")

flights = pd.concat(dfs, ignore_index=True)
logging.info(f"Total rows: {len(flights):,} | Columns: {list(flights.columns)}")

2026-04-09 16:01:33,137 - Found 12 CSV files
2026-04-09 16:01:33,518 - Loaded: aug-2024 - 619,025 rows
2026-04-09 16:01:33,748 - Loaded: sep-2024 - 582,622 rows
2026-04-09 16:01:33,998 - Loaded: april-2024 - 582,205 rows
2026-04-09 16:01:34,230 - Loaded: nov-2024 - 575,404 rows
2026-04-09 16:01:34,495 - Loaded: jun-2024 - 611,132 rows
2026-04-09 16:01:34,734 - Loaded: march-2024 - 591,767 rows
2026-04-09 16:01:34,964 - Loaded: oct-2024 - 615,497 rows
2026-04-09 16:01:35,201 - Loaded: may-2024 - 609,743 rows
2026-04-09 16:01:35,405 - Loaded: jan-2024 - 547,271 rows
2026-04-09 16:01:35,627 - Loaded: dec-2024 - 590,581 rows
2026-04-09 16:01:35,841 - Loaded: feb-2024 - 519,221 rows
2026-04-09 16:01:36,094 - Loaded: july-2024 - 634,613 rows
2026-04-09 16:01:36,408 - Total rows: 7,079,081 | Columns: ['YEAR', 'MONTH', 'DAY_OF_WEEK', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'ORIGIN_CITY_NAME', 'DEST', 'DEST_CITY_NAME', 'CRS_DEP_TIME', 'DEP_DELAY', 'DEP_DEL15', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY

## Merge Lookup Tables

In [22]:
# Merge L_MONTHS, L_WEEKDAYS, L_UNIQUE_CARRIERS to decode numeric codes
month_map = pd.read_csv("L_MONTHS.csv")
week_day_map = pd.read_csv("L_WEEKDAYS.csv")
airline_map = pd.read_csv("L_UNIQUE_CARRIERS.csv")

month_map = month_map.rename(columns={
    "Code":"MONTH",
    'Description': "MONTH_NAME"
})

week_day_map = week_day_map.rename(columns={
    "Code": "DAY_OF_WEEK",
    "Description": "DAY"
})

airline_map = airline_map.rename(columns={
    "Code": "OP_UNIQUE_CARRIER",
    "Description": "AIRLINE"
})

flights = flights.merge(month_map, on="MONTH", how="left")
flights = flights.merge(week_day_map, on="DAY_OF_WEEK", how="left")
flights = flights.merge(airline_map, on="OP_UNIQUE_CARRIER", how="left")

# sort by month and week day
flights = flights.sort_values(by=['MONTH', 'DAY_OF_WEEK'])
flights.head()

,YEAR,MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_DELAY,...,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,source_file,MONTH_NAME,DAY,AIRLINE
4787395,2024,1,1,9E,ABE,"Allentown/Bethlehem/Easton, PA",ATL,"Atlanta, GA",600,-11.0,...,692.0,NaN,NaN,NaN,NaN,NaN,jan-2024,January,Monday,Endeavor Air Inc.
4787396,2024,1,1,9E,ABE,"Allentown/Bethlehem/Easton, PA",ATL,"Atlanta, GA",600,-7.0,...,692.0,NaN,NaN,NaN,NaN,NaN,jan-2024,January,Monday,Endeavor Air Inc.
4787397,2024,1,1,9E,ABE,"Allentown/Bethlehem/Easton, PA",ATL,"Atlanta, GA",600,-5.0,...,692.0,NaN,NaN,NaN,NaN,NaN,jan-2024,January,Monday,Endeavor Air Inc.
4787398,2024,1,1,9E,ABE,"Allentown/Bethlehem/Easton, PA",ATL,"Atlanta, GA",600,-3.0,...,692.0,NaN,NaN,NaN,NaN,NaN,jan-2024,January,Monday,Endeavor Air Inc.
4787399,2024,1,1,9E,ABE,"Allentown/Bethlehem/Easton, PA",ATL,"Atlanta, GA",600,-2.0,...,692.0,NaN,NaN,NaN,NaN,NaN,jan-2024,January,Monday,Endeavor Air Inc.


## Log Extraction Stats

In [23]:
# Print: number of files loaded, rows per month, total rows
logging.info(f"Number of files loaded are {len(csv_files)}")
for month, count in flights.groupby("MONTH_NAME")["MONTH"].count().items():
    logging.info(f"{month}: {count:,} rows")
logging.info(f"Total rows: {len(flights):,}")

2026-04-09 16:01:46,091 - Number of files loaded are 12
2026-04-09 16:01:46,226 - April: 582,205 rows
2026-04-09 16:01:46,227 - August: 619,025 rows
2026-04-09 16:01:46,227 - December: 590,581 rows
2026-04-09 16:01:46,227 - February: 519,221 rows
2026-04-09 16:01:46,227 - January: 547,271 rows
2026-04-09 16:01:46,227 - July: 634,613 rows
2026-04-09 16:01:46,228 - June: 611,132 rows
2026-04-09 16:01:46,228 - March: 591,767 rows
2026-04-09 16:01:46,228 - May: 609,743 rows
2026-04-09 16:01:46,228 - November: 575,404 rows
2026-04-09 16:01:46,228 - October: 615,497 rows
2026-04-09 16:01:46,229 - September: 582,622 rows
2026-04-09 16:01:46,229 - Total rows: 7,079,081


## Save to Parquet

In [24]:
# Save combined DataFrame to flights_2024.parquet

output_dir = Path("data")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "flights_2024.parquet"

flights.to_parquet(output_path, index=False)
logging.info(f"Saved to {output_path}")


2026-04-09 16:03:10,898 - Saved to data/flights_2024.parquet
